# Deep Learning for Joint OD Word-Bit Prediction

We build a multi-label target over 16 words x 32 bits = 512 outputs.
Each output target y[w*32 + b] corresponds to whether bit b of CipherTextDiff_w is 1.
The model learns P(OD_word=w AND OD_bit=b | (ID_word, ID_bit, plaintext context)).

In [25]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import joblib

## 1) Load Dataset and Prepare Features

In [26]:
# Load the dataset
dataset = pd.read_csv('dataset.csv')
print(f"Dataset shape: {dataset.shape}")
print(f"Columns: {dataset.columns.tolist()}")
print(f"First few rows:")
print(dataset.head())

Dataset shape: (320000, 34)
Columns: ['IDWordIndex', 'IDBitIndex', 'PlainText_0', 'PlainText_1', 'PlainText_2', 'PlainText_3', 'PlainText_4', 'PlainText_5', 'PlainText_6', 'PlainText_7', 'PlainText_8', 'PlainText_9', 'PlainText_10', 'PlainText_11', 'PlainText_12', 'PlainText_13', 'PlainText_14', 'PlainText_15', 'CipherTextDiff_0', 'CipherTextDiff_1', 'CipherTextDiff_2', 'CipherTextDiff_3', 'CipherTextDiff_4', 'CipherTextDiff_5', 'CipherTextDiff_6', 'CipherTextDiff_7', 'CipherTextDiff_8', 'CipherTextDiff_9', 'CipherTextDiff_10', 'CipherTextDiff_11', 'CipherTextDiff_12', 'CipherTextDiff_13', 'CipherTextDiff_14', 'CipherTextDiff_15']
First few rows:
  IDWordIndex  IDBitIndex  PlainText_0 PlainText_1  PlainText_2  PlainText_3  \
0           c           0           65          6e           32           74   
1           c           1           65          6e           32           74   
2           c           2           65          6e           32           74   
3           c           3

In [27]:
# Encode input word symbol (e.g., 'c','d','e','f') to integer id
le_word_joint = LabelEncoder()
unique_words = dataset['IDWordIndex'].astype(str)
le_word_joint.fit(unique_words)
print(f"Unique input words: {le_word_joint.classes_}")

# Utility: hex string (up to 32-bit) -> 32-bit binary array
def hex_to_bits32(x: str) -> np.ndarray:
    # Ensure string, convert hex->int->binary string->pad->array of ints
    s = format(int(str(x), 16), '032b')
    return np.fromiter((1 if ch=='1' else 0 for ch in s), dtype=np.uint8, count=32)

Unique input words: ['c' 'd' 'e' 'f']


In [28]:
# Build X features: [input_word_id, input_bit]
# Build Y labels: 512-dimensional binary vector for all output word-bit combinations
X_list = []
Y_list = []

print("Processing dataset...")
# Iterate rows once; for each row create a 512-dim label
for idx, row in dataset.iterrows():
    if idx % 1000 == 0:
        print(f"Processed {idx} rows")
    
    input_word_id = le_word_joint.transform([str(row['IDWordIndex'])])[0]
    input_bit = int(row['IDBitIndex'])

    # Feature vector: [input_word_id, input_bit]
    X_list.append([input_word_id, input_bit])

    # Build 512 labels by concatenating bits of each CipherTextDiff_w (w=0..15)
    bits_all = []
    for w in range(16):
        bits = hex_to_bits32(row[f'CipherTextDiff_{w}'])  # shape (32,)
        bits_all.append(bits)
    y = np.concatenate(bits_all, axis=0)  # shape (512,)
    Y_list.append(y)

X_np = np.asarray(X_list, dtype=np.int64)            # shape (N, 2)
Y_np = np.asarray(Y_list, dtype=np.uint8)           # shape (N, 512)

print(f"\nFeature matrix shape: {X_np.shape}")
print(f"Label matrix shape: {Y_np.shape}")
print(f"Average number of active bits per sample: {Y_np.sum(axis=1).mean():.2f}")

Processing dataset...
Processed 0 rows
Processed 1000 rows
Processed 2000 rows
Processed 3000 rows
Processed 4000 rows
Processed 5000 rows
Processed 6000 rows
Processed 7000 rows
Processed 8000 rows
Processed 9000 rows
Processed 10000 rows
Processed 11000 rows
Processed 12000 rows
Processed 13000 rows
Processed 14000 rows
Processed 15000 rows
Processed 16000 rows
Processed 17000 rows
Processed 18000 rows
Processed 19000 rows
Processed 20000 rows
Processed 21000 rows
Processed 22000 rows
Processed 23000 rows
Processed 24000 rows
Processed 25000 rows
Processed 26000 rows
Processed 27000 rows
Processed 28000 rows
Processed 29000 rows
Processed 30000 rows
Processed 31000 rows
Processed 32000 rows
Processed 33000 rows
Processed 34000 rows
Processed 35000 rows
Processed 36000 rows
Processed 37000 rows
Processed 38000 rows
Processed 39000 rows
Processed 40000 rows
Processed 41000 rows
Processed 42000 rows
Processed 43000 rows
Processed 44000 rows
Processed 45000 rows
Processed 46000 rows
Proc

In [29]:
# Standardize numeric features for DL stability
scaler_joint = StandardScaler()
X_np_scaled = scaler_joint.fit_transform(X_np.astype(np.float32))

# Train/val split
# Use stratification based on whether any output bits are active
stratify_labels = (Y_np.sum(axis=1) > 0).astype(int)
X_train, X_val, y_train, y_val = train_test_split(
    X_np_scaled, Y_np, test_size=0.2, random_state=42, stratify=stratify_labels
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")

Training set: 256000 samples
Validation set: 64000 samples


## 2) PyTorch Dataset and DataLoader

In [30]:
class ODJointDataset(Dataset):
    def __init__(self, X: np.ndarray, Y: np.ndarray):
        self.X = torch.from_numpy(X).float()
        self.Y = torch.from_numpy(Y).float()
    
    def __len__(self):
        return self.X.shape[0]
    
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

batch_size = 1024
train_ds = ODJointDataset(X_train, y_train)
val_ds = ODJointDataset(X_val, y_val)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

Training batches: 250
Validation batches: 63


## 3) Define MLP Model

In [31]:
class ODJointMLP(nn.Module):
    def __init__(self, in_dim: int = 2, hidden: int = 256, out_dim: int = 512, p_drop: float = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(p_drop),
            nn.Linear(hidden, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(p_drop),
            nn.Linear(hidden, out_dim)
        )
    
    def forward(self, x):
        return self.net(x)  # logits for 512 labels

# Instantiate model, loss (BCE-with-logits), optimizer
model_joint = ODJointMLP(in_dim=X_train.shape[1], hidden=256, out_dim=512, p_drop=0.2)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model_joint.parameters(), lr=1e-3)

print(f"Model parameters: {sum(p.numel() for p in model_joint.parameters()):,}")

Model parameters: 198,144


## 4) Training Loop

In [32]:
def evaluate(model: nn.Module, data_loader: DataLoader):
    model.eval()
    total_loss = 0.0
    y_true_all = []
    y_prob_all = []
    
    with torch.no_grad():
        for xb, yb in data_loader:
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * xb.size(0)
            
            probs = torch.sigmoid(logits).cpu().numpy()
            y_prob_all.append(probs)
            y_true_all.append(yb.cpu().numpy())
    
    y_true = np.concatenate(y_true_all, axis=0)
    y_prob = np.concatenate(y_prob_all, axis=0)
    
    # Macro AUPRC/AUROC across 512 outputs
    try:
        ap = average_precision_score(y_true, y_prob, average='macro')
    except Exception:
        ap = np.nan
    
    try:
        auroc = roc_auc_score(y_true, y_prob, average='macro')
    except Exception:
        auroc = np.nan
    
    avg_loss = total_loss / len(data_loader.dataset)
    return avg_loss, ap, auroc

In [33]:
epochs = 10  # adjust upward if needed
best_val_ap = -1.0
best_state_dict = None

for epoch in range(1, epochs+1):
    model_joint.train()
    running = 0.0
    
    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model_joint(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        running += loss.item() * xb.size(0)
    
    train_loss = running / len(train_loader.dataset)
    val_loss, val_ap, val_auc = evaluate(model_joint, val_loader)
    
    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_mAP={val_ap:.4f} val_AUROC={val_auc:.4f}")
    
    # Track best by validation mAP
    if (not np.isnan(val_ap)) and val_ap > best_val_ap:
        best_val_ap = val_ap
        best_state_dict = {k: v.cpu().clone() for k, v in model_joint.state_dict().items()}

# Load best weights if improved
if best_state_dict is not None:
    model_joint.load_state_dict(best_state_dict)
    print(f"\nLoaded best model with validation mAP: {best_val_ap:.4f}")

c:\Users\alide\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\alide\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\alide\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\alide\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\alide\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y

Epoch 01 | train_loss=0.1939 val_loss=0.1709 val_mAP=0.1360 val_AUROC=nan


c:\Users\alide\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\alide\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\alide\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\alide\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\alide\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y

Epoch 02 | train_loss=0.1709 val_loss=0.1692 val_mAP=0.1408 val_AUROC=nan


c:\Users\alide\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\alide\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\alide\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\alide\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\alide\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y

...

c:\Users\alide\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\alide\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\alide\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\alide\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\alide\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y

Epoch 09 | train_loss=0.1656 val_loss=0.1641 val_mAP=0.1530 val_AUROC=nan


c:\Users\alide\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\alide\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\alide\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\alide\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\alide\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y

Epoch 10 | train_loss=0.1653 val_loss=0.1640 val_mAP=0.1531 val_AUROC=nan

Loaded best model with validation mAP: 0.1531


c:\Users\alide\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


## 5) Prediction Utilities

In [34]:
def predict_top_k_od(input_word: str, input_bit: int, top_k: int = 10):
    """
    Return top-k OD (word, bit) pairs with highest predicted probability.
    """
    x = np.asarray([[le_word_joint.transform([str(input_word)])[0], int(input_bit)]], dtype=np.int64)
    x = scaler_joint.transform(x.astype(np.float32))
    
    with torch.no_grad():
        logits = model_joint(torch.from_numpy(x).float())
        probs = torch.sigmoid(logits).cpu().numpy()[0]  # shape (512,)
    
    # Decode indices -> (word, bit)
    idxs = np.argsort(-probs)[:top_k]
    results = []
    for idx in idxs:
        od_word = idx // 32
        od_bit = idx % 32
        results.append((int(od_word), int(od_bit), float(probs[idx])))
    return results

In [35]:
# Example usage (prints top-5 OD pairs)
print("\nExample: top-5 OD pairs for ('c', 0)")
for w, b, p in predict_top_k_od('c', 0, top_k=5):
    print(f"  OD_word={w:2d}, OD_bit={b:2d}, prob={p:.4f}")


Example: top-5 OD pairs for ('c', 0)
  OD_word= 0, OD_bit=25, prob=0.9236
  OD_word= 1, OD_bit=31, prob=0.8159
  OD_word= 1, OD_bit=26, prob=0.7999
  OD_word=10, OD_bit=31, prob=0.7583
  OD_word= 2, OD_bit=27, prob=0.7537


## 6) Model Evaluation

In [36]:
# Evaluate model on validation set with concise summaries
val_loss, val_ap, val_auc = evaluate(model_joint, val_loader)
print(f"Final Validation: loss={val_loss:.4f}, mAP={val_ap:.4f}, AUROC={val_auc:.4f}")

# Quick sanity check across a few inputs
examples = [('c', 0), ('d', 5), ('e', 15)]
for iw, ib in examples:
    top = predict_top_k_od(iw, ib, top_k=5)
    print(f"\nInput ({iw}, {ib}) -> top-5 OD (word, bit, prob):")
    for w, b, p in top:
        print(f"  ({w:2d}, {b:2d}) p={p:.4f}")

c:\Users\alide\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\alide\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\alide\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\alide\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Users\alide\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y

Final Validation: loss=0.1640, mAP=0.1531, AUROC=nan

Input (c, 0) -> top-5 OD (word, bit, prob):
  ( 0, 25) p=0.9236
  ( 1, 31) p=0.8159
  ( 1, 26) p=0.7999
  (10, 31) p=0.7583
  ( 2, 27) p=0.7537

Input (d, 5) -> top-5 OD (word, bit, prob):
  ( 1, 28) p=0.8394
  ( 2, 29) p=0.8329
  ( 3, 30) p=0.8076
  ( 2, 26) p=0.7842
  (12, 29) p=0.7796

Input (e, 15) -> top-5 OD (word, bit, prob):
  ( 2, 26) p=0.9982
  ( 3, 27) p=0.9906
  (13, 27) p=0.9303
  (14, 28) p=0.8767
  (13, 24) p=0.8658


c:\Users\alide\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


## 7) Save Model

In [37]:
# Save model and encoders for later reuse
save_bundle = {
    'state_dict': model_joint.state_dict(),
    'scaler': scaler_joint,
    'label_encoder_word': le_word_joint,
    'input_dim': X_train.shape[1],
    'hidden': 256,
    'output_dim': 512,
}

# Torch model is saved as state_dict alongside scikit objects
joblib.dump(save_bundle, 'joint_od_predictor_bundle.pkl')
print("Saved: joint_od_predictor_bundle.pkl")

print("\nUsage note:")
print("To load later:")
print("  bundle = joblib.load('joint_od_predictor_bundle.pkl')")
print("  model = ODJointMLP(in_dim=bundle['input_dim'], hidden=bundle['hidden'], out_dim=bundle['output_dim'])")
print("  model.load_state_dict(bundle['state_dict'])")
print("  scaler = bundle['scaler']")
print("  le_word = bundle['label_encoder_word']")

Saved: joint_od_predictor_bundle.pkl

Usage note:
To load later:
  bundle = joblib.load('joint_od_predictor_bundle.pkl')
  model = ODJointMLP(in_dim=bundle['input_dim'], hidden=bundle['hidden'], out_dim=bundle['output_dim'])
  model.load_state_dict(bundle['state_dict'])
  scaler = bundle['scaler']
  le_word = bundle['label_encoder_word']


## 8) Advanced Prediction Utilities

In [38]:
def predict_od_matrix(input_word: str, input_bit: int) -> np.ndarray:
    """
    Return a (16, 32) matrix of P(OD_word=w, OD_bit=b | input) from DL model.
    """
    x = np.asarray([[le_word_joint.transform([str(input_word)])[0], int(input_bit)]], dtype=np.int64)
    x = scaler_joint.transform(x.astype(np.float32))
    
    with torch.no_grad():
        logits = model_joint(torch.from_numpy(x).float())
        probs = torch.sigmoid(logits).cpu().numpy()[0]  # shape (512,)
    
    return probs.reshape(16, 32)

def rank_output_words(prob_matrix: np.ndarray, agg: str = 'sum'):
    """
    Rank output words by aggregated bit probability.
    agg in {'sum','max','mean'} determines how to aggregate across 32 bits.
    Returns list of tuples (word, score).
    """
    if agg == 'sum':
        scores = prob_matrix.sum(axis=1)
    elif agg == 'max':
        scores = prob_matrix.max(axis=1)
    elif agg == 'mean':
        scores = prob_matrix.mean(axis=1)
    else:
        raise ValueError("agg must be one of {'sum','max','mean'}")
    
    ranking = sorted([(w, float(scores[w])) for w in range(16)], key=lambda x: x[1], reverse=True)
    return ranking

def rank_output_bits(prob_matrix: np.ndarray, output_word: int, top_k: int = 8):
    """
    Rank bits within a chosen output word. Returns top_k list of (bit, prob).
    """
    row = prob_matrix[output_word]
    idxs = np.argsort(-row)[:top_k]
    return [(int(b), float(row[b])) for b in idxs]

def predict_next_round_dl(input_word: str, input_bit: int, top_k_words: int = 5, bit_top_k: int = 8, agg: str = 'sum'):
    """
    DL-only interface to get best output words and bits.
    Returns dict with:
      - prob_matrix: (16,32) numpy array
      - top_words: list[(word, score)] of length top_k_words
      - top_bits_per_word: dict[word] -> list[(bit, prob)] of length bit_top_k
    """
    pm = predict_od_matrix(input_word, input_bit)
    words_ranked = rank_output_words(pm, agg=agg)[:top_k_words]
    bits_per = {w: rank_output_bits(pm, w, top_k=bit_top_k) for w, _ in words_ranked}
    
    return {
        'prob_matrix': pm,
        'top_words': words_ranked,
        'top_bits_per_word': bits_per,
    }

In [39]:
# DL-only example usage
examples = [('c', 0), ('d', 5), ('e', 15)]
for iw, ib in examples:
    res = predict_next_round_dl(iw, ib, top_k_words=3, bit_top_k=5, agg='sum')
    print(f"\nInput ({iw}, {ib})")
    print("Top output words (agg=sum):")
    for w, s in res['top_words']:
        print(f"  word={w:2d} score={s:.4f}")
    
    for w, _ in res['top_words']:
        print(f"  Best bits in word {w}:")
        for b, p in res['top_bits_per_word'][w]:
            print(f"    bit={b:2d} p={p:.4f}")


Input (c, 0)
Top output words (agg=sum):
  word= 2 score=4.4774
  word=13 score=4.4348
  word=10 score=4.3801
  Best bits in word 2:
    bit=27 p=0.7537
    bit=30 p=0.6822
    bit=24 p=0.6131
    bit=26 p=0.5726
    bit=31 p=0.5360
  Best bits in word 13:
    bit=25 p=0.7513
    bit=28 p=0.6452
    bit=30 p=0.6159
    bit=24 p=0.5440
    bit=27 p=0.5313
  Best bits in word 10:
    bit=31 p=0.7583
    bit=28 p=0.5889
    bit=26 p=0.5708
    bit=30 p=0.5265
    bit=25 p=0.5222

Input (d, 5)
Top output words (agg=sum):
  word= 3 score=4.3491
  word=14 score=4.3481
  word=12 score=4.2340
  Best bits in word 3:
    bit=30 p=0.8076
    bit=27 p=0.6710
    bit=25 p=0.5856
    bit=24 p=0.5423
    bit=26 p=0.4918
  Best bits in word 14:
    bit=28 p=0.7423
    bit=25 p=0.6564
    bit=31 p=0.6084
    bit=27 p=0.5279
    bit=30 p=0.5139
  Best bits in word 12:
    bit=29 p=0.7796
    bit=26 p=0.6553
    bit=25 p=0.5449
    bit=24 p=0.5004
    bit=28 p=0.4944

Input (e, 15)
Top output words (agg